In [16]:
from glob import glob
from cmcmultimodal.core.io import mat2nii
from pathlib import Path
import os

my_files = glob('/Users/Vasilis/CMC_data/Moe/PSOCT/Orientation/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)


In [ ]:
# TODO delete this section after all data have been flipped and analysis worked correctly
# Only to be used once!!!
import subprocess
import glob
from pathlib import Path

inp_path = Path('/Users/Vasilis/CMC_data/Moe/PSOCT')
modalities = []#['Cross', 'Retardance', 'Orientation', 'Reflectivity']

for mod in modalities:
    files = (inp_path / mod).glob('Slice_*_En*.nii.gz')
    for f in files:
        input  = f
        output = f

        subprocess.run(["fslswapdim",
                        input,
                        "-x", "y", "z",
                        output])
    
    files = (inp_path / mod / 'lowres').glob('Slice_*_En*.nii.gz')
    for f in files:
        input  = f
        output = f

        subprocess.run(["fslswapdim",
                        input,
                        "-x", "y", "z",
                        output])


In [3]:
# Mask Retardance by Cross
import subprocess
import os
from pathlib import Path
from fsl.wrappers import fslmaths#, LOAD

inp_path = Path('/Users/Vasilis/CMC_data/Moe/PSOCT')
target_mod  = 'Retardance'
mask_mod    = 'Cross'
out_path    = inp_path / (target_mod[:3] + '_masked_by_' + mask_mod[:3]) / 'lowres'
os.makedirs(out_path, exist_ok=True)

files = (inp_path / target_mod / 'lowres').glob('Slice_*_En*.nii.gz')
for f in files:
    mask_file = inp_path / mask_mod / 'lowres' / f.name.replace('EnR','EnCr')

    # mask = fslmaths(mask_file).bin().run(LOAD)

    fslmaths(f).mas(mask_file).run(out_path / f.name)


In [ ]:
# Process all MOE slides

from cmcmultimodal.core.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/CMC_data/Moe'
output_path = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test'
mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
seq_params = '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json'

my_data = psoct(Path(datadir), seq_params, psoct_reg_mod='Cross', mri_reg_mod='Ret_masked_by_Cro', verbose=True)

my_data.run_pipeline(other_images=['Cross', 'Retardance', 'Orientation'], 
                                    output_path=output_path, 
                                    mri_ref=mri_ref, 
                                    downsample=1, 
                                    bad_slides=[],
                                    fnirt=True)

In [ ]:
from pathlib import Path
from cmcmultimodal.utils.validate_results import compare_results_folder

ref_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret')
est_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret_new')

compare_results_folder(ref_path, est_path, subdir=True)

In [250]:
from fsl.wrappers.avwutils  import fslmerge
import glob, os
import numpy as np
from pathlib import Path
from fsl.wrappers                import flirt
from fsl.data.image              import Image
import dask.multiprocessing
import multiprocessing
import shutil

modality = 'Retardance'

out_file = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/' + modality[0:3] + '_slide_deck')
inp_files = sorted(glob.glob('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/' + modality + '/lowres/Slice*hdr.nii.gz'))
ref_files = sorted(glob.glob('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Ret_split/*.nii.gz'), reverse=True)

slide_numbers = [int(Path(f).name.split('_')[1]) for f in inp_files]

missing_slides = list(set(np.arange(min(slide_numbers), max(slide_numbers)+1)) - set(slide_numbers))
missing_slides = list(map(int, missing_slides))

slide_arr = np.array(slide_numbers)
raw_files = inp_files.copy()
for m in sorted(missing_slides):
    # slide_numbers = [int(Path(f).name.split('_')[1]) for f in inp_files]

    # nearest slide before
    before = slide_arr[(slide_arr - m) < 0]
    if before.size == 0:
        before = np.inf
    else:
        before = before[np.argmin(np.abs(before-m))]
    # nearest slide after
    after = slide_arr[(slide_arr - m) > 0]
    if after.size == 0:
        after = np.inf
    else:
        after = after[np.argmin(np.abs(after-m))]
    # If both are Inf (logically impossible but could happen if slide_numbers is empty), raise an error
    if np.isinf(before) and np.isinf(after):
        raise ValueError(f"No available slide before or after missing slide {m}")
    if np.abs(m-before) < np.abs(after-m):
        inp_files.insert(m-1, raw_files[np.where(slide_arr == before)[0][0]])
        ref_files[m-1] = ref_files[m-2]
    else:
        inp_files.insert(m-1, raw_files[np.where(slide_arr == after)[0][0]])#np.where(slide_arr == after)[0][0]])
        
for m in sorted(missing_slides, reverse=True):
    # nearest slide before
    before = slide_arr[(slide_arr - m) < 0]
    if before.size == 0:
        before = np.inf
    else:
        before = before[np.argmin(np.abs(before-m))]
    # nearest slide after
    after = slide_arr[(slide_arr - m) > 0]
    if after.size == 0:
        after = np.inf
    else:
        after = after[np.argmin(np.abs(after-m))]
    # If both are Inf (logically impossible but could happen if slide_numbers is empty), raise an error
    if np.isinf(before) and np.isinf(after):
        raise ValueError(f"No available slide before or after missing slide {m}")
    if np.abs(m-before) >= np.abs(after-m):
        ref_files[m-1] = ref_files[m]


os.makedirs(out_file.parent / (modality[0:3] +'_temp_files'), exist_ok=True)
jobs = []
for i in range(len(inp_files)):
    inp = inp_files[i]
    ref = ref_files[i]
    filename = 'vol_' + str(i).zfill(3) + '.nii.gz'
    out_filename = out_file.parent / (modality[0:3] + '_temp_files') / filename
    jobs.append(dask.delayed(flirt)(inp, ref, out=out_filename, usesqform=True, applyxfm=True, twod=True))
dask.compute(jobs)


out_files = sorted(glob.glob(str(out_file.parent / (modality[0:3] + '_temp_files/*.nii.gz'))), reverse=True)
fslmerge('y', out_file, *out_files)
shutil.rmtree(out_file.parent / (modality[0:3] + '_temp_files'))


In [ ]:
from cmc_hybrid.coordinate_mapping import get_pixgrid, in_cube, slide_vox_intersect, cube_vertices
from scipy.ndimage import map_coordinates
from cmc_hybrid import utils
from fsl.transform.fnirt import readFnirt
from fsl.transform.affine import concat, transform
from cmc_hybrid import coordinate_mapping as cm
from fsl.data.image import Image
from pathlib import Path
import numpy as np

testsPath   = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl_new/')
volume      = Image(testsPath / 'dtifit_FA')
slide_deck  = Image(testsPath / 'Ret_slide_deck')
slides = [Image(x) for x in [testsPath / 'Retardance/lowres/Slice_162_EnR_downsample_10_hdr', 
                             testsPath / 'Retardance/lowres/Slice_163_EnR_downsample_10_hdr',
                             testsPath / 'Retardance/lowres/Slice_164_EnR_downsample_10_hdr']]
slide2vol_warp = testsPath / 'PSOCT_to_MRI_warpfield.nii.gz'

vox    = [193, 182, 48]

def slide_vox_intersect(voxel, slide, volume, slide_deck, vox2world):
    """Test if voxel intersects with slide(s)

    WARNING!! This assumes that the data is Coronal!!!

    voxel      : list or array (single voxel)
    slide      : either Image object or list of Image objects
    volume_img : Image object

    Returns a bool or list of booleans
    """
    # If list provided, recursively run on list elements
    if type(slide) == list:
        return [slide_vox_intersect(voxel, s, volume, slide_deck, vox2world) for s in slide]
    # get transformed from header
    world2pix = slide.getAffine('world', 'voxel')
    pixel = vox2world.transform(voxel, 'voxel', 'world')

    # test if near slide using y-coord
    # calc distance between slide and centre of the voxel
    # test that it is smaller than sqrt(3)/2*edge

    dist = np.abs(transform(pixel, world2pix))[1] * slide.pixdim[1]
    return dist <= np.sqrt(3)/2. * np.max(volume.pixdim)

def get_pixgrid(voxel, slide, volume, slide_deck, vox2world):
    """Get a small grid of pixels surrounding a voxel
    This is a key function to avoid looking at all the coordinates
    of all the high resolution slide

    voxel : list or array (3 coordinates in voxel space)
    slide : Image object
    volume : Image object

    returns an array (Nx3)
    """
    # get transforms
    world2pix = slide.getAffine('world', 'voxel')

    pix2vox   = invert(vox2pix)

    # get voxel cube
    voxel_bounds     = cube_vertices(voxel, 1)
    voxel_bounds_pix = transform(vox2world.transform(voxel_bounds, 'voxel', 'world'), world2pix)

    imin, jmin = np.min(voxel_bounds_pix, axis=0)[::2]
    imax, jmax = np.max(voxel_bounds_pix, axis=0)[::2]

    pixgrid = np.array( np.meshgrid( np.arange(imin, imax+1), np.arange(jmin, jmax+1)) )
    pixgrid = np.reshape(pixgrid, (2, -1)).T
    pixgrid = np.stack( (pixgrid[:,0], 0*pixgrid[:,0], pixgrid[:,1]), axis=1 )

    pixgrid2vox = transform(pixgrid, pix2vox)

    return pixgrid2vox, pixgrid

if slide2vol_warp is not None:
    vox2world = readFnirt(slide2vol_warp, Image(slide_deck), Image(volume))
slide_masks = slide_vox_intersect(vox, slides, volume, slide_deck, vox2world)
#2) for each slide found, get the pixgrid (pixels surrounding the voxel)
pixgrid_all = []
voxgrid_all = []
pixgrid_data_all = []

for idx in range(len(slides)):
    if slide_masks[idx]:
        pixgrid2vox, pixgrid = get_pixgrid(vox, slides[idx], volume, slide_deck, slide2vol_warp)
        # select subset within the voxel
        mask        = in_cube(pixgrid2vox, 1, vox)
        pixgrid_all.extend(pixgrid[mask,:])
        voxgrid_all.extend(pixgrid2vox[mask,:])
        # get data
        data = utils.get_data(slides[idx])
        pixgrid_data_all.extend(map_coordinates(data, pixgrid[mask,:].T, order=0))


print(np.array(pixgrid_all).mean(axis=0))
print(pixgrid_all)

nan
[]


/var/folders/5k/jt__1pkj1j9bbztr70y_5_g40000gt/T/ipykernel_94217/3264984946.py:69: RuntimeWarning: Mean of empty slice.
  print(np.array(pixgrid_all).mean(axis=0))


In [ ]:
slide2vol_warp = testsPath / 'PSOCT_to_MRI_warpfield.nii.gz'
vox2world = readFnirt(slide2vol_warp, Image(slide_deck), Image(volume))

vox    = [193, 182, 48]
pixel = vox2world.transform(vox, 'voxel', 'world')

world2pix = slides[1].getAffine('world', 'voxel')

pix = transform(pixel, world2pix)

array([ 7.06932414e+02, -6.06203188e-03,  4.61440951e+02])

In [40]:
vox2world.transform(vox, 'voxel', 'voxel')

array([719.75842038, 163.99393797, 449.93008189])

In [ ]:
pixel

array([43.18550426, 40.99848449, 26.99580431])

In [ ]:
import subprocess
inp_folder = out_file.parent / (modality[0:3] +'_temp_files')
files = sorted(inp_folder.glob('*.nii.gz'))
subprocess.run(['fslmaths',
                files[0],
                '-mul', '0',
                str(out_file)])
cmd = ['fslmaths', str(out_file)]
for i in files:
    cmd.extend(['-add', str(i)])
cmd.append(str(out_file))
subprocess.run(cmd)

In [ ]:
from fsl.wrappers.avwutils  import fslmerge
import glob
from pathlib import Path
out_file = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_new/Ori_slide_deck')
inp_files = sorted(glob.glob('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_new/temp_files_2/Slice*hdr.nii.gz'), reverse=True)
fslmerge('y', out_file, *inp_files)

In [15]:
from fsl.utils.image.resample import resampleToReference
from fsl.data.image              import Image

src_filename = Image('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_new/Retardance/lowres/Slice_060_EnR_downsample_10_hdr.nii.gz')
tgt_filename = Image('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_new/Ret_slide_deck.nii.gz')
out_filename = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_new/resampled_slices/slide_060_new.nii.gz'
results = resampleToReference(src_filename, tgt_filename,)
Image(results[0], header = tgt_filename.header).save(out_filename)

In [29]:
from fsl.wrappers           import flirt

mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
outfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/dtifit_FA_in_Ret_PSOCT'
matfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/MRI_to_Ret_PSOCT.mat'

flirt(src=mri_ref, ref=out_file, out=outfile, omat=matfile, dof=12, interp='spline')


{}

In [3]:
import json
import numpy as np
from pathlib import Path

output_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross')

with open(output_path / "abs_mat.json") as f:
    data = json.load(f)

abs_mat = {int(k): np.array(v) for k, v in data.items()}